<a href="https://colab.research.google.com/github/gunwes16/data201-project1-classification/blob/main/analysis/notebooks/3.0-classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

url = "https://raw.githubusercontent.com/gunwes16/data201-project1-classification/refs/heads/main/data/processed/bank-clean%20(2).csv"
df = pd.read_csv(url)   # creates df; every later cell depends on it
print(df.shape)         # expect (45211, 17)

(45211, 17)


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier

In [8]:
y = df["y"]
X = df.drop(columns="y")
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 3), round(y_test.mean(), 3))

(36168, 16) (9043, 16)
0.117 0.117


In [9]:
def make_model(clf):
    prep = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore",
                              sparse_output=False), cat_cols)])
    return Pipeline([("prep", prep), ("clf", clf)])

In [10]:
lda = make_model(LinearDiscriminantAnalysis()).fit(X_train, y_train)
qda = make_model(QuadraticDiscriminantAnalysis(reg_param=0.1)).fit(X_train, y_train)
knn = make_model(KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
print("All four models fitted.")

All four models fitted.


In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}

rows = []
for name, clf in models.items():
    r = cross_validate(make_model(clf), X_train, y_train, cv=cv,
                       scoring=["accuracy", "precision", "recall", "f1"])
    rows.append({"model": name,
                 **{m: r["test_" + m].mean().round(3)
                    for m in ["accuracy", "precision", "recall", "f1"]}})
cv_results = pd.DataFrame(rows).sort_values("f1", ascending=False)
cv_results

,model,accuracy,precision,recall,f1
2,QDA,0.872,0.437,0.315,0.366
1,LDA,0.889,0.557,0.271,0.365
3,KNN (k=5),0.884,0.515,0.196,0.284
0,Logistic,0.892,0.643,0.174,0.274


In [12]:
for k in [1, 5, 15, 31, 51]:
    r = cross_validate(make_model(KNeighborsClassifier(n_neighbors=k)),
                       X_train, y_train, cv=cv, scoring=["accuracy", "recall", "f1"])
    print(f"K={k:>2}: accuracy {r['test_accuracy'].mean():.3f}  "
          f"recall {r['test_recall'].mean():.3f}  F1 {r['test_f1'].mean():.3f}")

K= 1: accuracy 0.842  recall 0.287  F1 0.297
K= 5: accuracy 0.884  recall 0.196  F1 0.284
K=15: accuracy 0.889  recall 0.141  F1 0.228
K=31: accuracy 0.889  recall 0.109  F1 0.187
K=51: accuracy 0.889  recall 0.087  F1 0.155
